# OneVoice V2 — publish verified fine-tuned models to Hugging Face

Uploads only the checkpoints that passed held-out evaluation. Repositories default to **private** until license/release review is completed. This notebook never deletes Google Drive artifacts; it writes a checksum receipt first.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
REPO = Path('/content/OneVoice')
ROOT = Path('/content/drive/MyDrive/OneVoice')

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub'], check=True)


In [ ]:
from getpass import getpass

# Token is kept only in this Colab runtime. Do not print it or save it to Drive.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('Hugging Face write token: ')
if not HF_TOKEN.startswith('hf_'):
    raise ValueError('Expected a Hugging Face token beginning with hf_')

PRIVATE_REPOS = True  # Change only after license/release review.
EN2VI_SOURCE = ROOT / 'models/envit5_finetuned_en2vi_v1/best'
SENSEVOICE_ROOT = ROOT / 'models/sensevoice_en_construction_v1'
SENSEVOICE_CHECKPOINT = SENSEVOICE_ROOT / 'model.pt.ep10'  # Held-out benchmark checkpoint.

for path in (EN2VI_SOURCE / 'config.json', SENSEVOICE_ROOT / 'config.yaml', SENSEVOICE_CHECKPOINT):
    if not path.is_file():
        raise FileNotFoundError(f'Missing verified source artifact: {path}')
print('EN->VI source:', EN2VI_SOURCE)
print('SenseVoice source:', SENSEVOICE_CHECKPOINT)


In [ ]:
import hashlib, json
from datetime import datetime, timezone
from huggingface_hub import HfApi

EN2VI_REPO = 'platypus123/onevoice-envit5-en-vi'
SENSEVOICE_REPO = 'platypus123/onevoice-sensevoice-en-construction-v1'
api = HfApi(token=HF_TOKEN)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

api.create_repo(EN2VI_REPO, repo_type='model', private=PRIVATE_REPOS, exist_ok=True)
api.upload_folder(
    repo_id=EN2VI_REPO, repo_type='model', folder_path=str(EN2VI_SOURCE),
    commit_message='Upload verified OneVoice EN-to-VI fine-tuned EnViT5 checkpoint',
)

api.create_repo(SENSEVOICE_REPO, repo_type='model', private=PRIVATE_REPOS, exist_ok=True)
sensevoice_card = '''---
license: other
tags:
- automatic-speech-recognition
- english
- construction
- onnx
---
# OneVoice SenseVoice EN construction checkpoint

Fine-tuned English ASR checkpoint used by OneVoice V2. The evaluated checkpoint is `model.pt` (uploaded from `model.pt.ep10`).

Held-out synthetic EN test: clean WER 0.374%; noisy WER 0.545%. This is a research/demo artifact trained on synthetic speech and construction-noise augmentation, not a real-site production claim. Use with the matching FunASR/SenseVoice training configuration.
'''
api.upload_file(path_or_fileobj=sensevoice_card.encode('utf-8'), path_in_repo='README.md', repo_id=SENSEVOICE_REPO, repo_type='model', commit_message='Add model card')
for source, target in ((SENSEVOICE_CHECKPOINT, 'model.pt'), (SENSEVOICE_ROOT / 'config.yaml', 'config.yaml'), (SENSEVOICE_ROOT / 'configuration.json', 'configuration.json')):
    if source.is_file():
        api.upload_file(path_or_fileobj=str(source), path_in_repo=target, repo_id=SENSEVOICE_REPO, repo_type='model', commit_message=f'Upload {target}')

receipt = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'private_repos': PRIVATE_REPOS,
    'uploads': {
        EN2VI_REPO: {path.relative_to(EN2VI_SOURCE).as_posix(): sha256(path) for path in EN2VI_SOURCE.rglob('*') if path.is_file()},
        SENSEVOICE_REPO: {
            'model.pt': sha256(SENSEVOICE_CHECKPOINT),
            **{target: sha256(source) for source, target in ((SENSEVOICE_ROOT / 'config.yaml', 'config.yaml'), (SENSEVOICE_ROOT / 'configuration.json', 'configuration.json')) if source.is_file()},
        },
    },
}
RECEIPT = ROOT / 'reports/model_publish_hf_v2/receipt.json'
RECEIPT.parent.mkdir(parents=True, exist_ok=True)
RECEIPT.write_text(json.dumps(receipt, indent=2), encoding='utf-8')
print('Uploaded:', EN2VI_REPO, 'and', SENSEVOICE_REPO)
print('Local checksum receipt:', RECEIPT)


In [ ]:
# Remote metadata verification. Keep Drive originals until this cell succeeds.
for repo_id, expected in receipt['uploads'].items():
    info = api.model_info(repo_id, files_metadata=True)
    remote = {item.rfilename: item for item in info.siblings}
    missing = sorted(set(expected) - set(remote))
    if missing:
        raise RuntimeError(f'{repo_id}: remote files missing: {missing}')
    for filename, local_hash in expected.items():
        lfs = getattr(remote[filename], 'lfs', None)
        remote_hash = getattr(lfs, 'sha256', None) if lfs else None
        if remote_hash and remote_hash != local_hash:
            raise RuntimeError(f'{repo_id}/{filename}: remote SHA-256 mismatch')
    print(f'PASS {repo_id}: {len(expected)} expected files listed remotely')

print('Publishing verified. Do not delete Drive originals until you explicitly choose retention/deletion.')
